# Identificacao de um alvo maritimo e estimativa de sua marcacao e distancia

TCC — 1T Matheus de Souza Rodrigues — C-ApA-Of-SA

O trabalho responde a duas perguntas sobre uma cena com varias embarcacoes:

> **qual delas e o alvo designado, e o que muda quando o detector e especializado para reconhece-lo?**

> **a partir da imagem, em que direcao e a que distancia o alvo se encontra, e com que incerteza?**

O alvo e a **lancha branca de cabine** da sequencia `kope75-00-00013780-00014195` do \ac{MODD2},
observada em **305 quadros** enquanto se aproxima. O distrator principal e o bote inflavel KP-1493,
que divide a cena com ela em boa parte do percurso.

| Parte | O que faz |
|---|---|
| 1 | configuracao |
| 2 | dados do MODD2 e a trilha do alvo |
| 3 | particao, data augmentation e dataset YOLO de duas classes |
| 4 | linha de base: o YOLO11 do COCO, sem treino |
| 5 | fine-tuning em `alvo` e `outros` |
| 6 | avaliacao: precisao, revocacao, F1, mAP, matriz de confusao e alcance |
| 7 | calibracao estereo da sequencia |
| 8 | marcacao e distancia ao longo da aproximacao |
| 9 | resumo, zip e download |

## O protocolo

A lancha aparece em uma unica cena, entao a particao do alvo e **temporal**: o trecho final da
trilha, em que ela esta proxima, vai para o treino; o intermediario, para a validacao; e o inicial,
em que ela esta distante, para o teste. Entre os blocos ficam quadros descartados, de modo que
treino e teste nao contenham quadros quase identicos.

| Conjunto | Quadros do alvo | Largura da caixa |
|---|---|---|
| treino | 153 | 110 a 274 px |
| validacao | 45 | 100 a 113 px |
| teste | 91 | 54 a 108 px |

A escolha de treinar com o alvo proximo e cobrar com o alvo distante e deliberada: ela torna o teste
mais dificil que o treino, e responde a pergunta util a um \ac{VSNT}, que precisa identificar cedo.
As demais 27 sequencias entram repartidas por cena, contribuindo com a classe `outros` e com quadros
de fundo, o que permite medir falso positivo em cenas nunca vistas.

## Limitacoes assumidas

- O alvo aparece em uma unica cena: treino e teste compartilham iluminacao, mar e fundo, e a
  generalizacao da classe `alvo` para outros cenarios nao e medida por este trabalho.
- Quadros consecutivos sao quase identicos, de modo que a amostra efetiva e menor que o numero de
  quadros sugere.
- O conjunto nao fornece distancia de referencia. A validacao da distancia e feita por analise de
  incerteza e por coerencia do comprimento implicito.

# Parte 1 — Configuracao

Todos os parametros ficam reunidos aqui. Nenhum numero magico aparece espalhado pelo resto do
notebook — o que se quiser variar, varia-se nesta celula.

In [ ]:
# ------------------------------------------------------------------
# Configuracao central
# ------------------------------------------------------------------
from pathlib import Path

# ---------- dados ----------
USAR_DRIVE   = True          # monta o Drive para guardar pesos e saidas entre sessoes
PARAR_SEM_GPU = True         # interrompe se o ambiente nao tiver GPU (treino em CPU e inviavel)
ID_DRIVE     = "1CIUXGnJnbXEh-j3SbwwTzpIly7csoI1t"   # zip do MODD2 no Drive
RAIZ_DADOS   = Path("/content/DATASET")
SAIDAS       = Path("/content/saidas"); SAIDAS.mkdir(exist_ok=True)
TREINOS      = Path("/content/treinos")

LARGURA_IMG, ALTURA_IMG = 1278, 958
LADO_USADO   = "L"           # so a camera esquerda tem anotacao
NOMES_CLASSES = ["alvo", "outros"]

# ---------- o alvo ----------
SEQUENCIA_ALVO = "kope75-00-00013780-00014195"
N_ALVO_ESPERADO = 305        # conferido na definicao do alvo; a Parte 2 verifica

# ---------- montagem da trilha ----------
# Criterio MISTO: aceita a ligacao por sobreposicao OU por proximidade de centro escalada pelo
# tamanho da caixa. O segundo caso e o que sustenta o alvo pequeno enquanto o VSNT guina — a
# sobreposicao sozinha perdia 46 quadros da fase distante, que e justamente a mais valiosa.
IOU_MINIMO     = 0.30
MAX_LACUNA     = 15          # quadros sem correspondencia antes de encerrar a trilha
DIST_CENTRO_MAX = 1.2        # deslocamento do centro, em multiplos do lado da caixa
RAZAO_LARGURA_MAX = 1.5      # variacao de largura tolerada na ligacao

# ---------- limpeza da trilha ----------
SALTO_MAXIMO   = 2.5         # corta onde a trilha troca de embarcacao
RAZAO_AREA_MAX = 3.0
MARGEM_BORDA   = 3           # descarta a caixa que encosta na borda: alvo cortado pelo enquadramento

# ---------- particao ----------
FRACAO_TESTE = 0.30
FRACAO_VALID = 0.15
BUFFER       = 8             # quadros descartados em cada fronteira
PASSO_OUTRAS = 3             # subamostragem das sequencias sem o alvo
SEMENTE      = 42

# ---------- treino ----------
PESOS   = "yolo11n.pt"
EPOCAS  = 40
IMGSZ   = 960
BATCH   = 8
ESCALA_AUG = 0.9             # sorteio de escala da propria Ultralytics, em tempo de treino

# ---------- data augmentation explicita, gravada em disco ----------
# Tres tecnicas simples, correspondentes a degradacoes reais da imagem no mar.
AUG_ATIVA   = True
AUG_FLIP    = True           # espelhamento: o alvo pode vir de qualquer bordo
AUG_BLUR    = (3, 5)         # filtro da media: maresia, respingo, perda de foco
AUG_AFASTAR = (0.60, 0.40)   # reduz a cena e reposiciona: simula o alvo mais distante
AUG_SO_ALVO = True           # aplicar apenas aos quadros da classe minoritaria

# ---------- avaliacao ----------
CONF_MIN  = 0.25
IOU_NMS   = 0.50
IOU_AVALIACAO   = 0.50
IOU_AVALIACAO_2 = 0.30       # as caixas do MODD2 sao mais justas que as de um detector do COCO

# ---------- estimativa de distancia ----------
BUSCA_DISPARIDADE = 260      # faixa horizontal de busca do gabarito, em pixels
TOLERANCIA_VERTICAL = 4      # tolerancia em linhas, para absorver residuo de retificacao
COMPRIMENTO_TIPICO_M = 6.5   # so para a checagem de coerencia, nunca para estimar

In [ ]:
# ------------------------------------------------------------------
# Dependencias
# ------------------------------------------------------------------
%pip install -q -U ultralytics gdown
import ultralytics, torch, cv2
print("ultralytics:", ultralytics.__version__, "| torch:", torch.__version__, "| opencv:", cv2.__version__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "sem GPU")
if DEVICE == "cpu" and PARAR_SEM_GPU:
    raise RuntimeError(
        "Sem GPU. Em CPU cada epoca leva cerca de meia hora e o treino nao termina.\\n"
        "No Colab: Ambiente de execucao -> Alterar o tipo -> T4 GPU.")

PASTA_DRIVE = None
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PASTA_DRIVE = Path("/content/drive/MyDrive/tcc_vsnt"); PASTA_DRIVE.mkdir(parents=True, exist_ok=True)
    TREINOS = PASTA_DRIVE / "treinos"
    print("[drive] pesos e saidas em", PASTA_DRIVE)
TREINOS.mkdir(parents=True, exist_ok=True)

## 1.1 — Obtendo o MODD2

O conjunto e baixado uma vez e reaproveitado enquanto a sessao durar. Sao usadas a versao
**retificada** das imagens e as anotacoes correspondentes — as duas precisam ser da mesma versao,
sob pena de as caixas nao casarem com as imagens.

In [ ]:
# ------------------------------------------------------------------
# Download e descompactacao
# ------------------------------------------------------------------
import subprocess, zipfile, os

def obter_zip():
    """Devolve o caminho do zip do MODD2, procurando na ordem mais barata.

    1) uma copia ja no /content (reexecucao na mesma sessao)
    2) uma copia no Drive montado — nao usa rede nenhuma
    3) download pelo gdown

    Sobre o gdown: a partir da versao 5 a opcao `--id` foi REMOVIDA. Passar o
    identificador na URL, com --fuzzy, e a forma que funciona nas versoes atuais.
    """
    zip_local = Path("/content/modd2.zip")
    if zip_local.exists() and zip_local.stat().st_size > 1e9:
        print(f"[local] zip ja presente ({zip_local.stat().st_size/1e9:.1f} GB)")
        return zip_local

    if PASTA_DRIVE is not None:
        for cand in list(PASTA_DRIVE.glob("*.zip")) + list(Path("/content/drive/MyDrive").glob("*.zip")):
            if cand.stat().st_size > 1e9:
                print(f"[drive] usando copia do Drive: {cand}")
                return cand

    url = f"https://drive.google.com/uc?id={ID_DRIVE}"
    for tentativa, args in enumerate([
            ["gdown", url, "-O", str(zip_local)],
            ["gdown", "--fuzzy", url, "-O", str(zip_local)],
            ["gdown", "--fuzzy", "--no-cookies", url, "-O", str(zip_local)]], 1):
        print(f"baixando o MODD2 (tentativa {tentativa}) ...")
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode == 0 and zip_local.exists() and zip_local.stat().st_size > 1e9:
            return zip_local
        print(f"  falhou: {(r.stderr or r.stdout or '').strip().splitlines()[-1:] or ['sem mensagem']}")

    raise RuntimeError(
        "Nao consegui baixar o MODD2.\n\n"
        "Causa mais comum: cota de download do Google Drive excedida, ou o arquivo nao esta\n"
        "compartilhado como 'qualquer pessoa com o link'.\n\n"
        "Solucao que sempre funciona: coloque o zip do conjunto na raiz do seu Google Drive\n"
        "(ou na pasta tcc_vsnt) e rode esta celula de novo — ela procura no Drive montado\n"
        "antes de tentar a rede, e nesse caso nao baixa nada.")

if not (RAIZ_DADOS / "video_data").exists():
    RAIZ_DADOS.mkdir(parents=True, exist_ok=True)
    zip_local = obter_zip()
    print("descompactando ...")
    with zipfile.ZipFile(zip_local) as z:
        # o zip foi feito no macOS e traz entradas __MACOSX/._* que nao servem para nada
        uteis = [n for n in z.namelist() if not n.startswith("__MACOSX") and "/._" not in n]
        z.extractall(RAIZ_DADOS, members=uteis)
    # o zip pode trazer uma pasta intermediaria; normaliza
    if not (RAIZ_DADOS / "video_data").exists():
        for sub in RAIZ_DADOS.iterdir():
            if (sub / "video_data").exists():
                for item in sub.iterdir():
                    item.rename(RAIZ_DADOS / item.name)
                break
assert (RAIZ_DADOS / "video_data").exists(), f"video_data nao encontrado em {RAIZ_DADOS}"
print("dados em", RAIZ_DADOS)
print("  sequencias:", len([d for d in (RAIZ_DADOS / "video_data").iterdir() if d.is_dir()]))

# Parte 2 — Os dados e a trilha do alvo

As anotacoes do \ac{MODD2} delimitam os obstaculos quadro a quadro, mas **nao atribuem identidade**
a eles: nada no conjunto diz que a caixa de um quadro corresponde a mesma embarcacao da caixa do
quadro seguinte. Recuperar essa identidade e o primeiro passo, e e o que permite designar uma
embarcacao especifica como alvo.

As caixas de quadros vizinhos sao ligadas em **trilhas**. O criterio usual para isso e a
sobreposicao entre as caixas, medida pela \ac{IoU}. Ele funciona bem para alvo grande e proximo,
mas falha no caso que mais interessa aqui: com o alvo a poucas dezenas de pixels e o \ac{VSNT}
guinando, a caixa se desloca na imagem mais do que a propria largura entre um quadro e o seguinte,
a sobreposicao cai a zero e a trilha se parte.

Por isso o criterio adotado e **misto**: a ligacao e aceita quando ha sobreposicao suficiente **ou**
quando o centro da caixa se desloca menos de `DIST_CENTRO_MAX` vezes o proprio tamanho e a largura
nao varia mais que `RAZAO_LARGURA_MAX`. A segunda condicao e o que sustenta a fase distante.

In [ ]:
# ------------------------------------------------------------------
# Indice dos quadros e leitura das anotacoes
# ------------------------------------------------------------------
import re, numpy as np, pandas as pd, scipy.io as sio

DIR_VIDEO = RAIZ_DADOS / "video_data"
DIR_GT    = RAIZ_DADOS / "annotationsV2_rectified"

CAMINHO_SEQ = {d.name: d / "framesRectified" for d in DIR_VIDEO.iterdir()
               if (d / "framesRectified").is_dir()}
CAMINHO_GT  = {d.name: d / "ground_truth" for d in DIR_GT.iterdir()
               if (d / "ground_truth").is_dir()}
SEQUENCIAS  = sorted(set(CAMINHO_SEQ) & set(CAMINHO_GT))
assert SEQUENCIA_ALVO in SEQUENCIAS, f"sequencia do alvo ausente em {RAIZ_DADOS}"

PADRAO = re.compile(r"(\d+)([LR])\.jpe?g$", re.IGNORECASE)
registros = []
for seq in SEQUENCIAS:
    for cam in sorted(CAMINHO_SEQ[seq].glob(f"*{LADO_USADO}.jp*g")):
        m = PADRAO.search(cam.name)
        if not m:
            continue
        mat = CAMINHO_GT[seq] / (cam.stem + ".mat")
        if mat.exists():
            registros.append({"sequencia": seq, "indice": int(m.group(1)),
                              "caminho": str(cam), "mat": str(mat)})
df_imgs = pd.DataFrame(registros).sort_values(["sequencia", "indice"]).reset_index(drop=True)

def ler_anotacao(caminho_mat):
    """Caixas (x1, y1, x2, y2) em pixels. Cast para float64: alguns arquivos vem em uint8."""
    try:
        a = sio.loadmat(str(caminho_mat))["annotations"][0, 0]
        ob = a["obstacles"]
    except Exception:
        return []
    caixas = []
    if ob.size:
        for linha in np.atleast_2d(np.asarray(ob, dtype=np.float64)):
            x, y, w, h = linha[:4]
            if w > 0 and h > 0:
                caixas.append((float(x), float(y), float(x + w), float(y + h)))
    return caixas

anotacoes = {r["caminho"]: ler_anotacao(r["mat"]) for _, r in df_imgs.iterrows()}

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

print(f"{len(SEQUENCIAS)} sequencias | {len(df_imgs)} quadros | "
      f"{sum(len(v) for v in anotacoes.values())} obstaculos anotados")

In [ ]:
# ------------------------------------------------------------------
# Trilhas por criterio misto, limpeza e selecao do alvo
# ------------------------------------------------------------------
def afinidade(anterior, nova):
    """Quanto a caixa nova casa com a ultima da trilha.

    Devolve a IoU; mas se o centro se deslocou pouco em relacao ao tamanho e a largura e
    compativel, devolve um valor acima do limiar mesmo sem sobreposicao. E o que atravessa a
    fase distante, onde a guinada do VSNT desloca a caixa mais que a propria largura.
    """
    v = iou(anterior, nova)
    ca = ((anterior[0] + anterior[2]) / 2, (anterior[1] + anterior[3]) / 2)
    cn = ((nova[0] + nova[2]) / 2, (nova[1] + nova[3]) / 2)
    lado = max(anterior[2] - anterior[0], anterior[3] - anterior[1], 1.0)
    la, ln = anterior[2] - anterior[0], nova[2] - nova[0]
    if (np.hypot(cn[0] - ca[0], cn[1] - ca[1]) / lado <= DIST_CENTRO_MAX and
            max(la, ln) / max(min(la, ln), 1e-6) <= RAZAO_LARGURA_MAX):
        return max(v, IOU_MINIMO + 0.01)
    return v

def montar_trilhas(seq):
    trilhas, ativas = [], []
    for _, linha in df_imgs[df_imgs["sequencia"] == seq].iterrows():
        idx, cam = linha["indice"], linha["caminho"]
        caixas = anotacoes[cam]
        pares = sorted(((afinidade(t["quadros"][-1][1], c), i, j)
                        for i, t in enumerate(ativas) for j, c in enumerate(caixas)), reverse=True)
        usadas_t, usadas_c = set(), set()
        for v, i, j in pares:
            if v < IOU_MINIMO or i in usadas_t or j in usadas_c:
                continue
            ativas[i]["quadros"].append((idx, caixas[j], cam))
            usadas_t.add(i); usadas_c.add(j)
        for j, c in enumerate(caixas):
            if j not in usadas_c:
                nova = {"sequencia": seq, "quadros": [(idx, c, cam)]}
                ativas.append(nova); trilhas.append(nova)
        ativas = [t for t in ativas if idx - t["quadros"][-1][0] <= MAX_LACUNA]
    return trilhas

def limpar(quadros):
    """Corta onde a trilha troca de embarcacao e devolve o maior trecho continuo."""
    cortes = [0]
    for k in range(1, len(quadros)):
        (i0, a, _), (i1, b, _) = quadros[k - 1], quadros[k]
        lacuna = max(1, i1 - i0)
        ca = ((a[0]+a[2])/2, (a[1]+a[3])/2); cb = ((b[0]+b[2])/2, (b[1]+b[3])/2)
        lado = max(a[2]-a[0], a[3]-a[1], 1.0)
        salto = np.hypot(cb[0]-ca[0], cb[1]-ca[1]) / lado
        aa = (a[2]-a[0]) * (a[3]-a[1]); ab = (b[2]-b[0]) * (b[3]-b[1])
        if salto > SALTO_MAXIMO * lacuna or max(aa, ab) / max(min(aa, ab), 1e-6) > RAZAO_AREA_MAX * lacuna:
            cortes.append(k)
    cortes.append(len(quadros))
    i0, i1 = max(zip(cortes[:-1], cortes[1:]), key=lambda p: p[1] - p[0])
    return quadros[i0:i1]

def cortar_bordas(quadros):
    """Descarta os quadros em que a caixa encosta na borda: o alvo esta cortado pelo enquadramento
    e tanto a largura quanto o centro deixam de representar a embarcacao."""
    def inteira(c):
        return (c[0] > MARGEM_BORDA and c[1] > MARGEM_BORDA and
                c[2] < LARGURA_IMG - MARGEM_BORDA and c[3] < ALTURA_IMG - MARGEM_BORDA)
    return [q for q in quadros if inteira(q[1])]

candidatas = [cortar_bordas(limpar(t["quadros"])) for t in montar_trilhas(SEQUENCIA_ALVO)]
candidatas = [c for c in candidatas if c]
quadros_alvo = max(candidatas, key=len)
CAIXA_ALVO = {cam: caixa for _, caixa, cam in quadros_alvo}
larguras = [c[2] - c[0] for _, c, _ in quadros_alvo]

print("=" * 70)
print("TRILHA DO ALVO")
print("=" * 70)
print(f"Sequencia : {SEQUENCIA_ALVO}")
print(f"Quadros   : {len(quadros_alvo)}  (q{quadros_alvo[0][0]} a q{quadros_alvo[-1][0]})")
print(f"Largura   : {min(larguras):.0f} a {max(larguras):.0f} px | mediana {np.median(larguras):.0f}")
print(f"Outras trilhas desta cena: {len(candidatas) - 1}")

# trava: a definicao do alvo esta registrada em alvo_definicao.json e nao deve mudar em silencio
if len(quadros_alvo) != N_ALVO_ESPERADO:
    raise RuntimeError(
        f"A trilha do alvo deu {len(quadros_alvo)} quadros, e nao os {N_ALVO_ESPERADO} registrados "
        f"na definicao do alvo. Algum parametro da montagem mudou — confira IOU_MINIMO, "
        f"MAX_LACUNA, DIST_CENTRO_MAX e RAZAO_LARGURA_MAX antes de prosseguir.")
print(f"\\n[ok] {N_ALVO_ESPERADO} quadros, conforme a definicao registrada do alvo.")

## 2.1 — Vendo o alvo

Duas figuras antes de seguir. A primeira e o **perfil de aproximacao**: a largura da caixa do alvo
ao longo do tempo, com os tres blocos da particao marcados. Ela mostra de uma vez a decisao
metodologica central — o treino fica na parte alta da curva, o teste na parte baixa.

A segunda e a **folha de contatos**: o alvo recortado em varios instantes, do mais distante ao mais
proximo. E a conferencia de que a trilha acompanha sempre a mesma embarcacao.

In [ ]:
# ------------------------------------------------------------------
# Perfil de aproximacao, com os blocos da particao
# ------------------------------------------------------------------
import matplotlib.pyplot as plt

n = len(quadros_alvo)
n_teste, n_valid = int(n * FRACAO_TESTE), int(n * FRACAO_VALID)
faixa_teste  = (0, n_teste)
faixa_valid  = (n_teste + BUFFER, n_teste + BUFFER + n_valid)
faixa_treino = (n_teste + BUFFER + n_valid + BUFFER, n)

idx  = [q[0] for q in quadros_alvo]
larg = [q[1][2] - q[1][0] for q in quadros_alvo]

AZUL, LARANJA, AGUA, TINTA = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b"
fig, ax = plt.subplots(figsize=(12, 4.6))
for (a, b), cor, nome in ((faixa_teste, AGUA, "teste"), (faixa_valid, LARANJA, "validacao"),
                          (faixa_treino, AZUL, "treino")):
    ax.axvspan(idx[a], idx[min(b, n) - 1], color=cor, alpha=0.13, zorder=0)
    ax.text((idx[a] + idx[min(b, n) - 1]) / 2, 0.95, nome, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=10, color=TINTA)
ax.plot(idx, larg, color=TINTA, linewidth=1.8, zorder=3)
ax.set_xlabel("quadro da sequencia"); ax.set_ylabel("largura da caixa do alvo (px)")
ax.set_title("Perfil de aproximacao do alvo e os blocos da particao", fontsize=12)
ax.grid(axis="y", alpha=0.18)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig(SAIDAS / "perfil_alvo.png", dpi=130, bbox_inches="tight"); plt.show()

for nome, (a, b) in (("teste", faixa_teste), ("validacao", faixa_valid), ("treino", faixa_treino)):
    l = larg[a:b]
    print(f"  {nome:10s}: {len(l):3d} quadros | {min(l):3.0f}-{max(l):3.0f} px | mediana {np.median(l):3.0f}")
print(f"  {'descarte':10s}: {2 * BUFFER:3d} quadros entre os blocos")

In [ ]:
# ------------------------------------------------------------------
# Folha de contatos do alvo, do mais distante ao mais proximo
# ------------------------------------------------------------------
from PIL import Image
import matplotlib.patches as patches

def recorte(cam, caixa, margem=2.1, minimo=170):
    img = Image.open(cam).convert("RGB")
    x1, y1, x2, y2 = caixa
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    lado = max((x2 - x1) * margem, (y2 - y1) * margem, minimo)
    corte = (max(0, cx - lado/2), max(0, cy - lado/2),
             min(img.width, cx + lado/2), min(img.height, cy + lado/2))
    return img.crop([int(v) for v in corte])

ordem = sorted(quadros_alvo, key=lambda q: q[1][2] - q[1][0])
sel = [ordem[k] for k in np.linspace(0, len(ordem) - 1, 18).astype(int)]
cols = 6; rows = 3
fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 2.9 * rows), squeeze=False)
for ax, (i, c, cam) in zip(axes.ravel(), sel):
    ax.imshow(recorte(cam, c))
    ax.set_title(f"q{i} | {c[2]-c[0]:.0f} px", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("O alvo ao longo da aproximacao — da aparicao mais distante a mais proxima", fontsize=13)
plt.tight_layout(); plt.savefig(SAIDAS / "folha_contatos_alvo.png", dpi=125, bbox_inches="tight"); plt.show()

In [ ]:
# ------------------------------------------------------------------
# O alvo e o distrator, lado a lado no mesmo quadro
# ------------------------------------------------------------------
duplos = [(i, c, cam) for i, c, cam in quadros_alvo
          if len([o for o in anotacoes[cam] if iou(o, c) <= 0.9 and o[2] - o[0] >= 40]) > 0]
print(f"Quadros em que o alvo divide a cena com outra embarcacao: "
      f"{len(duplos)} de {len(quadros_alvo)} ({len(duplos)/len(quadros_alvo):.0%})")

if duplos:
    sel = [duplos[k] for k in np.linspace(0, len(duplos) - 1, 4).astype(int)]
    fig, axes = plt.subplots(2, len(sel), figsize=(3.1 * len(sel), 6.6), squeeze=False)
    for col, (i, c, cam) in enumerate(sel):
        outros = [o for o in anotacoes[cam] if iou(o, c) <= 0.9 and o[2] - o[0] >= 40]
        maior = max(outros, key=lambda o: o[2] - o[0])
        for linha, (caixa, rotulo, cor) in enumerate([(c, "ALVO", "red"),
                                                      (maior, "distrator", "deepskyblue")]):
            ax = axes[linha, col]
            ax.imshow(recorte(cam, caixa, margem=1.8, minimo=150))
            ax.set_title(f"{rotulo}\n{caixa[2]-caixa[0]:.0f} px", fontsize=9, color=cor, weight="bold")
            ax.axis("off")
    plt.suptitle("A tarefa do trabalho, em uma imagem: distinguir o de cima do de baixo", fontsize=13)
    plt.tight_layout(); plt.savefig(SAIDAS / "alvo_vs_distrator.png", dpi=130, bbox_inches="tight"); plt.show()

# Parte 3 — Particao, data augmentation e dataset

A trilha e cortada em tres blocos na ordem do tempo, com `BUFFER` quadros descartados em cada
fronteira. As demais sequencias sao repartidas **inteiras** entre os conjuntos: os obstaculos
anotados viram `outros`, e os quadros sem obstaculo algum viram fundo puro, que e o que ensina o
modelo a nao inventar embarcacao em reflexo de agua.

## Data augmentation

A classe `alvo` esta em uma fracao pequena dos quadros de treino. Repetir o mesmo quadro varias
vezes nao resolve: tres copias identicas carregam a mesma informacao de uma so, e o que mudam e o
peso, nao o que o modelo aprende.

Por isso cada quadro do alvo entra uma vez como original e mais uma vez por **tecnica de
augmentation**, cada copia diferente da outra. Sao tres tecnicas simples, escolhidas por
corresponderem a degradacoes que a imagem realmente sofre no mar:

| Tecnica | O que faz | O que simula |
|---|---|---|
| flip horizontal | espelha a imagem | o alvo aproximando por qualquer bordo |
| filtro da media 3x3 e 5x5 | cada pixel vira a media dos vizinhos | maresia, respingo na lente, perda de foco |
| reducao 0,60 e 0,40 | encolhe a cena e a reposiciona sobre fundo neutro | o alvo mais distante |

A terceira e a que ataca a fraqueza especifica deste trabalho. O alvo e treinado entre 110 e 274 px
e cobrado entre 54 e 108 px. Reduzir a cena inteira nao deixa o alvo apenas menor: deixa-o menor
**e com menos detalhe**, que e o que a distancia de fato faz. Encolher so a caixa produziria um alvo
pequeno e nitido, que nao existe no mar.

Tudo e gerado **em disco, antes do treino**, e apenas para o treino. Validacao e teste ficam com os
quadros originais, e a celula interrompe a execucao se encontrar uma imagem aumentada fora do
treino. O flip horizontal da propria Ultralytics e desligado, porque passou a ser feito aqui —
espelhar duas vezes de forma independente produziria augmentacao que nao aparece em contagem nenhuma.

A geracao de imagens sinteticas por redes adversarias foi considerada e nao empregada: exige treinar
uma rede geradora a parte e levanta a questao de quanto a embarcacao sintetizada corresponde ao alvo
real. Fica registrada como evolucao natural desta etapa.

In [ ]:
# ------------------------------------------------------------------
# Particao: temporal para o alvo, por cena para o resto
# ------------------------------------------------------------------
import random

cam_teste  = [q[2] for q in quadros_alvo[faixa_teste[0]:faixa_teste[1]]]
cam_valid  = [q[2] for q in quadros_alvo[faixa_valid[0]:faixa_valid[1]]]
cam_treino = [q[2] for q in quadros_alvo[faixa_treino[0]:]]

outras = [s for s in SEQUENCIAS if s != SEQUENCIA_ALVO]
rng = random.Random(SEMENTE); rng.shuffle(outras)
n_tr = round(len(outras) * 0.60); n_va = max(1, round(len(outras) * 0.15))
grupos = {"train": outras[:n_tr], "valid": outras[n_tr:n_tr+n_va], "test": outras[n_tr+n_va:]}

particao = {"train": list(cam_treino), "valid": list(cam_valid), "test": list(cam_teste)}
for sp, seqs in grupos.items():
    for seq in seqs:
        particao[sp].extend(list(df_imgs[df_imgs["sequencia"] == seq]["caminho"])[::PASSO_OUTRAS])

# quadros da cena do alvo fora da trilha: seguem a particao temporal mais proxima
cams_cena = list(df_imgs[df_imgs["sequencia"] == SEQUENCIA_ALVO]["caminho"])
pos = {c: i for i, c in enumerate(cams_cena)}
lim_te = pos[quadros_alvo[faixa_teste[1] - 1][2]]
lim_va = pos[quadros_alvo[faixa_valid[1] - 1][2]]
for c in cams_cena[::PASSO_OUTRAS]:
    if c in CAIXA_ALVO:
        continue
    particao["test" if pos[c] <= lim_te else ("valid" if pos[c] <= lim_va else "train")].append(c)

print("=" * 74); print("PARTICAO"); print("=" * 74)
for sp in ("train", "valid", "test"):
    n_a = sum(1 for c in set(particao[sp]) if c in CAIXA_ALVO)
    print(f"  {sp:6s}: {len(set(particao[sp])):5d} quadros | {n_a:4d} com o alvo | "
          f"{len(grupos[sp]):2d} sequencias sem o alvo")

In [ ]:
# ------------------------------------------------------------------
# Dataset no formato YOLO, com a augmentation gravada em disco
# ------------------------------------------------------------------
import shutil, os, yaml

DIR_DATASET = RAIZ_DADOS / "modd2_alvo"
if DIR_DATASET.exists():
    shutil.rmtree(DIR_DATASET)
for sp in ("train", "valid", "test"):
    (DIR_DATASET / sp / "images").mkdir(parents=True, exist_ok=True)
    (DIR_DATASET / sp / "labels").mkdir(parents=True, exist_ok=True)

def caixas_do_quadro(cam):
    """Caixas anotadas em pixels, com a classe. 0 = alvo, 1 = outros."""
    alvo = CAIXA_ALVO.get(cam)
    return [(tuple(c), 0 if (alvo is not None and iou(c, alvo) > 0.9) else 1)
            for c in anotacoes[cam]]

def linhas_de(caixas, larg, alt):
    linhas = []
    for (x1, y1, x2, y2), classe in caixas:
        cx, cy = ((x1 + x2) / 2) / larg, ((y1 + y2) / 2) / alt
        w, h = (x2 - x1) / larg, (y2 - y1) / alt
        if w <= 0 or h <= 0:
            continue
        linhas.append(f"{classe} {min(max(cx,0),1):.6f} {min(max(cy,0),1):.6f} "
                      f"{min(w,1):.6f} {min(h,1):.6f}")
    return linhas

def linhas_yolo(cam):
    return linhas_de(caixas_do_quadro(cam), LARGURA_IMG, ALTURA_IMG)

# ---------- as tres tecnicas ----------
def aplicar_aug(img, caixas, tecnica, valor, rng):
    """Transforma imagem E caixas juntas. Caixa que nao acompanha e rotulo errado,
    que e pior que nao aumentar nada."""
    if tecnica == "flip":
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        L = img.width
        caixas = [((L - x2, y1, L - x1, y2), c) for (x1, y1, x2, y2), c in caixas]
    elif tecnica == "blur":
        # filtro da media: cada pixel vira a media da vizinhanca valor x valor.
        # A geometria nao muda, entao as caixas seguem iguais.
        img = Image.fromarray(cv2.blur(np.array(img), (valor, valor)))
    elif tecnica == "afastar":
        # reduz a cena inteira e a reposiciona sobre fundo de cor media: o alvo fica menor
        # E perde detalhe, que e o que a distancia faz. So encolher a caixa nao serviria.
        L, A = img.size
        nl, na = max(1, int(L * valor)), max(1, int(A * valor))
        pequena = img.resize((nl, na), Image.BILINEAR)
        cor = tuple(int(v) for v in np.array(img).reshape(-1, 3).mean(axis=0))
        fundo = Image.new("RGB", (L, A), cor)
        ox, oy = rng.randint(0, L - nl), rng.randint(0, A - na)
        fundo.paste(pequena, (ox, oy)); img = fundo
        caixas = [((x1*valor+ox, y1*valor+oy, x2*valor+ox, y2*valor+oy), c)
                  for (x1, y1, x2, y2), c in caixas]
    else:
        raise ValueError(f"tecnica desconhecida: {tecnica}")
    return img, caixas

def montar_variantes():
    """A lista de variantes que cada quadro do alvo vai gerar."""
    v = []
    if AUG_FLIP:
        v.append(("flip", None))
    for k in AUG_BLUR:
        v.append(("blur", k))
    for s in AUG_AFASTAR:
        v.append(("afastar", s))
    return v

VARIANTES = montar_variantes() if AUG_ATIVA else []
rotulo_var = lambda t, p: t if p is None else f"{t}{p}"

# ATENCAO: sequencias diferentes reutilizam os mesmos numeros de quadro. Sem o prefixo da
# sequencia, um arquivo sobrescreveria o outro e imagens sumiriam em silencio.
seq_por_caminho = dict(zip(df_imgs["caminho"], df_imgs["sequencia"]))
contagem, n_alvo = {}, {}

def gravar(sp, cam_str, base):
    cam = Path(cam_str)
    destino = DIR_DATASET / sp / "images" / (base + cam.suffix)
    try:
        os.symlink(cam, destino)          # link em vez de copia: rapido e nao duplica 1,7 GB
    except OSError:
        shutil.copy(cam, destino)
    linhas = linhas_yolo(cam_str)
    (DIR_DATASET / sp / "labels" / (base + ".txt")).write_text(
        "\n".join(linhas) + ("\n" if linhas else ""))

def gravar_aug(sp, cam_str, base, tecnica, valor, rng):
    img, caixas = aplicar_aug(Image.open(cam_str).convert("RGB"),
                              caixas_do_quadro(cam_str), tecnica, valor, rng)
    img.save(DIR_DATASET / sp / "images" / (base + ".jpg"), quality=92)
    linhas = linhas_de(caixas, img.width, img.height)
    (DIR_DATASET / sp / "labels" / (base + ".txt")).write_text(
        "\n".join(linhas) + ("\n" if linhas else ""))

rng_aug = random.Random(SEMENTE)
por_tecnica, total_aug = {}, 0
for sp, quadros in particao.items():
    quadros = sorted(set(quadros))
    n_aug = 0
    for cam_str in quadros:
        base = f"{seq_por_caminho[cam_str]}__{Path(cam_str).stem}"
        gravar(sp, cam_str, base)
        # augmentation: SO no treino, e so nos quadros da classe minoritaria
        if sp == "train" and VARIANTES and (cam_str in CAIXA_ALVO or not AUG_SO_ALVO):
            for tecnica, valor in VARIANTES:
                rot = rotulo_var(tecnica, valor)
                gravar_aug(sp, cam_str, f"{base}_aug-{rot}", tecnica, valor, rng_aug)
                por_tecnica[rot] = por_tecnica.get(rot, 0) + 1
                n_aug += 1
    total_aug += n_aug
    contagem[sp] = len(quadros) + n_aug
    n_alvo[sp] = sum(1 for c in quadros if c in CAIXA_ALVO)
    n_img = len(list((DIR_DATASET / sp / "images").glob("*.jp*g")))
    assert n_img == contagem[sp], f"{sp}: {contagem[sp]} pedidos, {n_img} gravados - colisao de nomes"

# ---------- trava: nenhuma imagem aumentada pode existir fora do treino ----------
for sp in ("valid", "test"):
    intrusos = sorted((DIR_DATASET / sp / "images").glob("*_aug-*"))
    if intrusos:
        raise RuntimeError(
            f"VAZAMENTO DE AUGMENTATION: {len(intrusos)} imagens aumentadas em '{sp}'. "
            f"Validacao e teste tem de conter apenas quadros originais. Ex.: {intrusos[0].name}")

YAML_DATASET = DIR_DATASET / "data.yaml"
YAML_DATASET.write_text(yaml.safe_dump(
    {"path": str(DIR_DATASET), "train": "train/images", "val": "valid/images",
     "test": "test/images", "nc": 2, "names": NOMES_CLASSES}, sort_keys=False))

print("=" * 74); print("DATASET YOLO"); print("=" * 74)
for sp in ("train", "valid", "test"):
    cxs = sum(len(linhas_yolo(c)) for c in set(particao[sp]))
    print(f"  {sp:6s}: {contagem[sp]:5d} imagens | {n_alvo[sp]:4d} quadros com o alvo | {cxs:5d} caixas")
if total_aug:
    print(f"\n  Data augmentation — {total_aug} imagens geradas, todas no treino:")
    for rot, n in por_tecnica.items():
        print(f"    {rot:12s}: {n:4d}")
    b = n_alvo["train"]
    print(f"\n  A classe alvo sai de {b} para {b * (1 + len(VARIANTES))} imagens de treino "
          f"({1 + len(VARIANTES)}x), e cada copia e diferente da outra.")
    print("  Validacao e teste ficam so com os quadros originais (conferido acima).")
print("-" * 74); print("data.yaml:", YAML_DATASET)

## 3.1 — As tecnicas de augmentation, vistas

A figura aplica cada tecnica ao mesmo quadro do alvo, com a caixa desenhada **ja transformada**.
E a conferencia que importa: augmentacao que move a imagem e esquece o rotulo ensina o modelo a
procurar o alvo no lugar errado, e o erro nao aparece em metrica nenhuma — aparece como desempenho
pior sem explicacao.

In [ ]:
# ------------------------------------------------------------------
# Figura didatica: o mesmo quadro sob cada tecnica
# ------------------------------------------------------------------
if not VARIANTES:
    print("Augmentation desligada (AUG_ATIVA = False).")
else:
    cam_demo = max((c for c in particao["train"] if c in CAIXA_ALVO),
                   key=lambda c: CAIXA_ALVO[c][2] - CAIXA_ALVO[c][0])
    rng_demo = random.Random(SEMENTE)
    base_img, base_cxs = Image.open(cam_demo).convert("RGB"), caixas_do_quadro(cam_demo)
    paineis = [("original", base_img, base_cxs)]
    for tecnica, valor in VARIANTES:
        im, cx = aplicar_aug(base_img.copy(), list(base_cxs), tecnica, valor, rng_demo)
        paineis.append((rotulo_var(tecnica, valor), im, cx))

    cols = 3; rows = int(np.ceil(len(paineis) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5.6 * cols, 4.4 * rows), squeeze=False)
    for ax, (nome, im, cxs) in zip(axes.ravel(), paineis):
        ax.imshow(im)
        for (x1, y1, x2, y2), classe in cxs:
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                           edgecolor="red" if classe == 0 else "deepskyblue", linewidth=2))
        la = next((x2 - x1 for (x1, _, x2, _), c in cxs if c == 0), None)
        ax.set_title(nome + (f" | alvo {la:.0f} px" if la else ""), fontsize=11,
                     fontweight="bold" if nome == "original" else "normal")
        ax.axis("off")
    for ax in axes.ravel()[len(paineis):]:
        ax.axis("off")
    plt.suptitle("Data augmentation — a mesma cena sob cada tecnica.  vermelho = alvo | azul = outros",
                 fontsize=14)
    plt.tight_layout(); plt.savefig(SAIDAS / "augmentation.png", dpi=110, bbox_inches="tight"); plt.show()

    l0 = next((x2 - x1 for (x1, _, x2, _), c in base_cxs if c == 0), 0)
    print(f"Alvo no quadro original: {l0:.0f} px")
    for nome, _, cxs in paineis[1:]:
        L = next((x2 - x1 for (x1, _, x2, _), c in cxs if c == 0), None)
        if L:
            dentro = "   <- entra na faixa do teste" if 54 <= L <= 108 else ""
            print(f"  {nome:12s}: {L:5.0f} px{dentro}")

## 3.2 — A augmentacao aproximou o treino do teste?

A secao anterior mostrou **o que** cada tecnica faz com uma imagem. Esta responde se o conjunto
inteiro se moveu, que e a afirmacao que de fato precisa ser sustentada.

O problema, em uma frase: o alvo e treinado entre 110 e 274 px e cobrado entre 54 e 108 px. As duas
faixas **nao se tocam**. O modelo nunca viu a lancha do tamanho em que vai ser testado.

A figura le as larguras da classe `alvo` **dos rotulos gravados em disco** — nao das caixas em
memoria — e separa em tres distribuicoes. O numero impresso abaixo, a fracao das caixas de treino
que cai dentro da faixa do teste antes e depois, e a resposta objetiva.

In [ ]:
# ------------------------------------------------------------------
# Distribuicao de tamanho do alvo: treino original, treino aumentado e teste
# ------------------------------------------------------------------
def larguras_alvo(sp, aumentadas=None):
    saida = []
    for lab in sorted((DIR_DATASET / sp / "labels").glob("*.txt")):
        if aumentadas is not None and ("_aug-" in lab.stem) != aumentadas:
            continue
        for linha in lab.read_text().splitlines():
            p = linha.split()
            if p and int(p[0]) == 0:
                saida.append(float(p[3]) * LARGURA_IMG)
    return np.array(saida)

tr_orig, tr_aug, te = larguras_alvo("train", False), larguras_alvo("train", True), larguras_alvo("test")
if len(tr_aug) == 0:
    print("Sem variantes aumentadas. Figura pulada.")
else:
    lo, hi = te.min(), te.max()
    dentro = lambda v: float(np.mean((v >= lo) & (v <= hi))) if len(v) else 0.0
    tr_todo = np.concatenate([tr_orig, tr_aug])
    series = [("treino, original", tr_orig, AZUL), ("treino, aumentado", tr_aug, LARANJA),
              ("teste", te, AGUA)]
    fig, ax = plt.subplots(figsize=(11, 5.2))
    bins = np.linspace(0, max(tr_todo.max(), te.max()) * 1.05, 34)
    ax.axvspan(lo, hi, color=AGUA, alpha=0.10, zorder=0)
    ax.annotate(f"faixa cobrada no teste\n{lo:.0f}-{hi:.0f} px", xy=((lo+hi)/2, 0.97),
                xycoords=("data", "axes fraction"), ha="center", va="top", fontsize=9, color="#52514e")
    for nome, v, cor in series:
        ax.hist(v, bins=bins, histtype="stepfilled", color=cor, alpha=0.22, zorder=2)
        ax.hist(v, bins=bins, histtype="step", color=cor, linewidth=2, label=nome, zorder=3)
    ax.set_xlabel("largura da caixa do alvo (pixels)"); ax.set_ylabel("numero de caixas")
    ax.set_title("Tamanho aparente do alvo: o que o modelo treina contra o que ele e cobrado",
                 fontsize=12, pad=14)
    ax.margins(y=0.16); ax.legend(frameon=False, loc="upper right"); ax.grid(axis="y", alpha=0.18)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    plt.tight_layout(); plt.savefig(SAIDAS / "distribuicao_tamanho.png", dpi=130, bbox_inches="tight"); plt.show()

    print("=" * 78)
    print(f"{'conjunto':22s} {'caixas':>7s} {'minimo':>8s} {'mediana':>8s} {'maximo':>8s} {'na faixa do teste':>19s}")
    print("-" * 78)
    for nome, v, _ in series:
        if len(v):
            print(f"{nome:22s} {len(v):7d} {v.min():8.0f} {np.median(v):8.0f} {v.max():8.0f} {dentro(v):18.1%}")
    print("-" * 78)
    print(f"{'treino, tudo junto':22s} {len(tr_todo):7d} {tr_todo.min():8.0f} {np.median(tr_todo):8.0f} "
          f"{tr_todo.max():8.0f} {dentro(tr_todo):18.1%}")
    print(f"\nCaixas de treino dentro da faixa do teste ({lo:.0f}-{hi:.0f} px):")
    print(f"  antes da augmentacao : {dentro(tr_orig):6.1%}")
    print(f"  depois               : {dentro(tr_todo):6.1%}   (ganho {dentro(tr_todo)-dentro(tr_orig):+.1%})")
    if dentro(tr_todo) - dentro(tr_orig) <= 0.01:
        print("\n[aviso] A augmentacao nao aproximou as distribuicoes. Reveja AUG_AFASTAR.")

## 3.3 — Conferindo o dataset gravado

A ultima conferencia antes do treino le os arquivos **do disco** — abre o `.txt` do rotulo e desenha
a caixa a partir dele, em vez de confiar no que esta em memoria. E o que pega erro de escrita, de
conversao de coordenada e de colisao de nome.

In [ ]:
# ------------------------------------------------------------------
# Amostra do dataset montado, com os rotulos lidos do disco
# ------------------------------------------------------------------
COR_CLASSE = {0: "red", 1: "deepskyblue"}

def ler_label(caminho_txt):
    caixas = []
    for linha in Path(caminho_txt).read_text().splitlines():
        if not linha.strip():
            continue
        c, cx, cy, w, h = linha.split()
        c, cx, cy, w, h = int(c), float(cx), float(cy), float(w), float(h)
        caixas.append((c, (cx-w/2)*LARGURA_IMG, (cy-h/2)*ALTURA_IMG,
                       (cx+w/2)*LARGURA_IMG, (cy+h/2)*ALTURA_IMG))
    return caixas

def painel_dataset(sp, filtro, titulo, arquivo):
    labels = [l for l in sorted((DIR_DATASET / sp / "labels").glob("*.txt")) if filtro(l)]
    if not labels:
        print(f"[info] nada para mostrar: {titulo}"); return
    random.Random(SEMENTE).shuffle(labels)
    esc = labels[:6]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8.6), squeeze=False)
    for ax, lab in zip(axes.ravel(), esc):
        img_path = next((DIR_DATASET / sp / "images").glob(lab.stem + ".*"))
        ax.imshow(np.array(Image.open(img_path).convert("RGB")))
        for c, x1, y1, x2, y2 in ler_label(lab):
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                           edgecolor=COR_CLASSE[c], linewidth=2))
            ax.text(x1, max(12, y1-5), NOMES_CLASSES[c], color=COR_CLASSE[c], fontsize=9, weight="bold")
        ax.set_title(lab.stem[-38:], fontsize=8); ax.axis("off")
    for ax in axes.ravel()[len(esc):]:
        ax.axis("off")
    plt.suptitle(titulo + "   (vermelho: alvo | azul: outros)", fontsize=14)
    plt.tight_layout(); plt.savefig(SAIDAS / arquivo, dpi=100, bbox_inches="tight"); plt.show()

tem_alvo = lambda l: any(c == 0 for c, *_ in ler_label(l))
painel_dataset("train", lambda l: tem_alvo(l) and "_aug-" not in l.stem,
               "TREINO — quadros originais com o alvo (fase proxima)", "dataset_train_alvo.png")
painel_dataset("train", lambda l: tem_alvo(l) and "_aug-" in l.stem,
               "TREINO — variantes aumentadas, lidas do disco", "dataset_train_aug.png")
painel_dataset("test",  tem_alvo,
               "TESTE — quadros com o alvo (fase distante)", "dataset_test_alvo.png")
painel_dataset("train", lambda l: not tem_alvo(l),
               "TREINO — amostra geral: outras cenas e fundo", "dataset_train_geral.png")

# Parte 4 — Antes do treino: o que o modelo generico faz

O YOLO11 pre-treinado no COCO conhece a classe generica `boat`. A pergunta e outra: ele sabe **qual**
barco importa? Nao sabe, e nem poderia — nada no COCO define alvo de interesse. O que se mede aqui e
o quanto o detector generico encontra o alvo, para servir de linha de base.

O avaliador e escrito na mao, com casamento guloso por \ac{IoU}, para que o antes e o depois passem
exatamente pelo mesmo criterio. O desempenho e reportado tambem por faixa de tamanho aparente, que
e onde mora a resposta sobre alcance de identificacao.

In [ ]:
# ------------------------------------------------------------------
# Avaliador proprio (mesmo criterio para antes e depois)
# ------------------------------------------------------------------
from ultralytics import YOLO

FAIXAS = [(0, 70, "< 70 px"), (70, 100, "70-100 px"), (100, 150, "100-150 px"), (150, 9999, "> 150 px")]
def faixa_de(largura):
    for lo, hi, nome in FAIXAS:
        if lo <= largura < hi:
            return nome
    return FAIXAS[-1][2]

def detectar(modelo, cam, classes_aceitas=None, conf=None):
    """Detecoes normalizadas: lista de (caixa_xyxy, nome_classe, confianca)."""
    r = modelo.predict(cam, conf=conf if conf is not None else CONF_MIN, iou=IOU_NMS,
                       imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    saida = []
    for b in r.boxes:
        nome = r.names[int(b.cls[0])]
        if classes_aceitas and nome not in classes_aceitas:
            continue
        saida.append((tuple(b.xyxy[0].tolist()), nome, float(b.conf[0])))
    return saida

def avaliar_alvo(modelo, cams, classes_aceitas=None, classe_alvo=None, iou_min=IOU_AVALIACAO):
    """Recall do alvo por faixa de tamanho.

    classe_alvo=None   -> basta a caixa ser encontrada por qualquer classe aceita (modelo generico)
    classe_alvo='alvo' -> a deteccao tambem precisa ter sido classificada como alvo
    """
    por_faixa = {nome: [0, 0] for _, _, nome in FAIXAS}
    acertos_classe = 0
    for cam in cams:
        gt = CAIXA_ALVO[cam]
        f = faixa_de(gt[2] - gt[0]); por_faixa[f][1] += 1
        casadas = [(iou(d[0], gt), d) for d in detectar(modelo, cam, classes_aceitas)]
        casadas = [c for c in casadas if c[0] >= iou_min]
        if casadas:
            por_faixa[f][0] += 1
            if classe_alvo and max(casadas)[1][1] == classe_alvo:
                acertos_classe += 1
    return por_faixa, acertos_classe

def tabela_recall(por_faixa, titulo):
    print(titulo); print("-" * 48)
    te = tt = 0
    for _, _, nome in FAIXAS:
        e, t = por_faixa[nome]; te += e; tt += t
        if t:
            print(f"  {nome:11s}: {e:4d}/{t:4d} quadros   recall {e/t:6.1%}")
    if tt:
        print(f"  {'total':11s}: {te:4d}/{tt:4d} quadros   recall {te/tt:6.1%}")
    return (te / tt) if tt else 0.0

In [ ]:
# ------------------------------------------------------------------
# Linha de base: YOLO11 do COCO sobre o trecho de teste
# ------------------------------------------------------------------
modelo_coco = YOLO(PESOS)
CLASSES_MARITIMAS = {"boat", "ship"}

print("=" * 70); print("ANTES DO TREINO — YOLO11 pre-treinado no COCO"); print("=" * 70)
antes_50, _ = avaliar_alvo(modelo_coco, cam_teste, CLASSES_MARITIMAS, iou_min=IOU_AVALIACAO)
r_antes_50 = tabela_recall(antes_50, f"\nDeteccao do alvo (IoU >= {IOU_AVALIACAO})")
antes_30, _ = avaliar_alvo(modelo_coco, cam_teste, CLASSES_MARITIMAS, iou_min=IOU_AVALIACAO_2)
r_antes_30 = tabela_recall(antes_30, f"\nDeteccao do alvo (IoU >= {IOU_AVALIACAO_2})")
print("\n[nota] O modelo generico nao separa alvo de outra embarcacao: quando acerta, marca 'boat'.")
print("       Por isso aqui so existe deteccao, e nao identificacao do alvo.")

# Parte 5 — Fine-tuning

O transfer learning parte dos pesos do COCO e reajusta o modelo para duas classes. Os pesos ficam no
Drive, identificados por uma etiqueta que descreve a configuracao — assim uma reexecucao com os
mesmos parametros reaproveita o treino, e uma reexecucao com parametros diferentes treina de novo
em vez de carregar o modelo errado em silencio.

In [ ]:
# ------------------------------------------------------------------
# Fine-tuning do YOLO11 em duas classes
# ------------------------------------------------------------------
TAG = f"{Path(PESOS).stem}_{IMGSZ}px_esc{ESCALA_AUG}_aug{len(VARIANTES)}_alvo{N_ALVO_ESPERADO}"
destino_pesos = (PASTA_DRIVE / f"best_{TAG}.pt") if PASTA_DRIVE else Path(f"/content/best_{TAG}.pt")

if destino_pesos.exists():
    print(f"[reaproveitando] pesos ja treinados para esta configuracao: {destino_pesos.name}")
    modelo_alvo = YOLO(str(destino_pesos))
else:
    print(f"[treinando] {TAG}\n")
    m = YOLO(PESOS)
    m.train(data=str(YAML_DATASET), epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH,
            device=0 if DEVICE == "cuda" else "cpu",
            project=str(TREINOS), name="yolo11_alvo", exist_ok=True,
            patience=10, pretrained=True, seed=SEMENTE, verbose=True,
            scale=ESCALA_AUG,
            fliplr=0.0)   # flip desligado aqui: passou a ser explicito na Parte 3
    melhor = Path(m.trainer.best)
    shutil.copy(melhor, destino_pesos)
    print(f"\n[guardado] {destino_pesos}")
    modelo_alvo = YOLO(str(destino_pesos))
print("classes do modelo:", modelo_alvo.names)

# Parte 6 — Depois do treino

Quatro medidas, em ordem de importancia para a pergunta do trabalho.

A primeira e o **mAP** por classe, que nao depende da escolha de um limiar de confianca. A segunda e
a comparacao completa de **precisao, revocacao e F1** entre o modelo generico e o especializado,
sobre exatamente os mesmos quadros — revocacao sozinha engana, porque um modelo que marca tudo tem
revocacao alta. A terceira e o **alcance**: como o desempenho varia com o tamanho aparente do alvo.
A quarta, e a que responde a pergunta central, e a **matriz de confusao** entre `alvo`, `outros` e
fundo: quando o modelo ve a embarcacao de interesse, ele a identifica como alvo ou a confunde com
outra embarcacao?

In [ ]:
# ------------------------------------------------------------------
# Metricas padrao no conjunto de teste
# ------------------------------------------------------------------
metricas = modelo_alvo.val(data=str(YAML_DATASET), split="test", imgsz=IMGSZ,
                           device=0 if DEVICE == "cuda" else "cpu", verbose=False)
print("=" * 70); print("METRICAS NO CONJUNTO DE TESTE"); print("=" * 70)
print(f"{'classe':10s} {'P':>8s} {'R':>8s} {'mAP@50':>8s} {'mAP@50-95':>10s}")
for i, nome in enumerate(NOMES_CLASSES):
    p, r, m50, m = metricas.box.class_result(i)
    print(f"{nome:10s} {p:8.3f} {r:8.3f} {m50:8.3f} {m:10.3f}")
print(f"{'media':10s} {metricas.box.mp:8.3f} {metricas.box.mr:8.3f} "
      f"{metricas.box.map50:8.3f} {metricas.box.map:10.3f}")

In [ ]:
# ------------------------------------------------------------------
# Precisao, revocacao e F1 dos dois modelos, sobre os mesmos quadros
# ------------------------------------------------------------------
def avaliar_completo(modelo, cams, classes_aceitas=None, iou_min=IOU_AVALIACAO_2):
    """TP, FP e FN contando TODAS as caixas anotadas, nao so o alvo."""
    TP = FP = FN = 0
    for cam in cams:
        gts = list(anotacoes[cam])
        det = detectar(modelo, cam, classes_aceitas)
        usadas = set()
        for g in gts:
            pares = sorted(((iou(d[0], g), k) for k, d in enumerate(det) if k not in usadas), reverse=True)
            if pares and pares[0][0] >= iou_min:
                TP += 1; usadas.add(pares[0][1])
            else:
                FN += 1
        FP += len(det) - len(usadas)
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    return {"TP": TP, "FP": FP, "FN": FN, "P": P, "R": R,
            "F1": 2 * P * R / (P + R) if P + R else 0.0}

antes_full  = avaliar_completo(modelo_coco, cam_teste, CLASSES_MARITIMAS)
depois_full = avaliar_completo(modelo_alvo, cam_teste, None)

print("=" * 74); print(f"COMPARACAO COMPLETA — conjunto de teste (IoU >= {IOU_AVALIACAO_2})"); print("=" * 74)
print(f"{'modelo':24s} {'TP':>5s} {'FP':>5s} {'FN':>5s} {'precisao':>9s} {'revocacao':>10s} {'F1':>7s}")
print("-" * 74)
for nome, r in (("YOLO11 COCO", antes_full), ("YOLO11 especializado", depois_full)):
    print(f"{nome:24s} {r['TP']:5d} {r['FP']:5d} {r['FN']:5d} {r['P']:9.3f} {r['R']:10.3f} {r['F1']:7.3f}")
print("-" * 74)
print(f"{'ganho de F1':24s} {'':5s} {'':5s} {'':5s} {'':9s} {'':10s} {depois_full['F1']-antes_full['F1']:+7.3f}")
print("\n[nota] Revocacao alta com precisao baixa significa marcar demais. E por isso que a")
print("       comparacao usa F1, e nao revocacao isolada.")

fig, ax = plt.subplots(figsize=(8.4, 4.4))
rot = ["precisao", "revocacao", "F1"]; x = np.arange(3); w = 0.36
ax.bar(x - w/2, [antes_full[k] for k in ("P","R","F1")], w, label="YOLO11 COCO", color=AZUL)
ax.bar(x + w/2, [depois_full[k] for k in ("P","R","F1")], w, label="YOLO11 especializado", color=LARANJA)
for i, k in enumerate(("P","R","F1")):
    ax.text(i - w/2, antes_full[k] + .015, f"{antes_full[k]:.3f}", ha="center", fontsize=9)
    ax.text(i + w/2, depois_full[k] + .015, f"{depois_full[k]:.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(rot); ax.set_ylim(0, 1.12)
ax.set_title("Antes e depois da adaptacao ao dominio", fontsize=12)
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.18)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig(SAIDAS / "comparativo_metricas.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ------------------------------------------------------------------
# Alcance: deteccao e identificacao do alvo por faixa de tamanho
# ------------------------------------------------------------------
depois_50, cls_50 = avaliar_alvo(modelo_alvo, cam_teste, None, "alvo", IOU_AVALIACAO)
depois_30, cls_30 = avaliar_alvo(modelo_alvo, cam_teste, None, "alvo", IOU_AVALIACAO_2)
print("=" * 70); print("DEPOIS DO TREINO — modelo especializado"); print("=" * 70)
r_depois_50 = tabela_recall(depois_50, f"\nDeteccao do alvo (IoU >= {IOU_AVALIACAO})")
r_depois_30 = tabela_recall(depois_30, f"\nDeteccao do alvo (IoU >= {IOU_AVALIACAO_2})")
print(f"\nIdentificado como classe 'alvo': {cls_30}/{len(cam_teste)} ({cls_30/len(cam_teste):.1%})")

nomes = [n for _, _, n in FAIXAS if antes_30[n][1] > 0]
va = [antes_30[n][0]/antes_30[n][1] for n in nomes]
vd = [depois_30[n][0]/depois_30[n][1] for n in nomes]
fig, ax = plt.subplots(figsize=(9, 4.4))
x = np.arange(len(nomes)); w = 0.36
ax.bar(x - w/2, va, w, label="YOLO11 COCO", color=AZUL)
ax.bar(x + w/2, vd, w, label="YOLO11 especializado", color=LARANJA)
for i in range(len(nomes)):
    ax.text(i - w/2, va[i] + .02, f"{va[i]:.0%}", ha="center", fontsize=9)
    ax.text(i + w/2, vd[i] + .02, f"{vd[i]:.0%}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([f"{n}\n({antes_30[n][1]} quadros)" for n in nomes])
ax.set_ylabel("revocacao do alvo"); ax.set_ylim(0, 1.15)
ax.set_title(f"Alcance de deteccao por tamanho aparente (IoU >= {IOU_AVALIACAO_2})", fontsize=12)
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.18)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig(SAIDAS / "recall_por_faixa.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ------------------------------------------------------------------
# Matriz de confusao: alvo x outros x fundo
# ------------------------------------------------------------------
def matriz_confusao(modelo, cams, iou_min=IOU_AVALIACAO_2):
    M = np.zeros((3, 3), dtype=int)     # linhas: alvo, outros, fundo | colunas: alvo, outros, nao det.
    for cam in cams:
        alvo = CAIXA_ALVO.get(cam)
        gts = [(c, 0 if (alvo is not None and iou(c, alvo) > 0.9) else 1) for c in anotacoes[cam]]
        det = detectar(modelo, cam)
        usadas = set()
        for caixa, classe_gt in gts:
            pares = sorted(((iou(d[0], caixa), k) for k, d in enumerate(det) if k not in usadas), reverse=True)
            if pares and pares[0][0] >= iou_min:
                k = pares[0][1]; usadas.add(k)
                M[classe_gt, 0 if det[k][1] == "alvo" else 1] += 1
            else:
                M[classe_gt, 2] += 1
        for k, d in enumerate(det):
            if k not in usadas:
                M[2, 0 if d[1] == "alvo" else 1] += 1
    return M

M = matriz_confusao(modelo_alvo, cam_teste)
print("=" * 70); print(f"MATRIZ DE CONFUSAO — teste (IoU >= {IOU_AVALIACAO_2})"); print("=" * 70)
cab = "verdade \\ predito"
print(f"{cab:>20s} {'alvo':>8s} {'outros':>8s} {'nao det.':>10s}")
for i, nome in enumerate(["alvo", "outros", "fundo"]):
    print(f"{nome:>20s} {M[i,0]:8d} {M[i,1]:8d} {M[i,2]:10d}")
if M[0].sum():
    print(f"\nDas {M[0].sum()} aparicoes do alvo no teste:")
    print(f"  identificadas como alvo : {M[0,0]:4d} ({M[0,0]/M[0].sum():.1%})")
    print(f"  confundidas com outros  : {M[0,1]:4d} ({M[0,1]/M[0].sum():.1%})  <- a pergunta central")
    print(f"  nao detectadas          : {M[0,2]:4d} ({M[0,2]/M[0].sum():.1%})")
np.savetxt(SAIDAS / "matriz_confusao.csv", M, fmt="%d", delimiter=",")

fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(M, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(["alvo", "outros", "nao detectado"])
ax.set_yticks(range(3)); ax.set_yticklabels(["alvo", "outros", "fundo"])
ax.set_xlabel("predito pelo modelo"); ax.set_ylabel("verdade (anotacao)")
lim = M.max() / 2 if M.max() else 1
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{M[i,j]}", ha="center", va="center", fontsize=14,
                color="white" if M[i, j] > lim else "black")
ax.set_title(f"Matriz de confusao — teste (IoU >= {IOU_AVALIACAO_2})", fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(SAIDAS / "matriz_confusao.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ------------------------------------------------------------------
# Figura antes x depois, no alvo distante e no alvo proximo
# ------------------------------------------------------------------
ref_perto = max(cam_treino, key=lambda c: CAIXA_ALVO[c][2] - CAIXA_ALVO[c][0])
ref_longe = min(cam_teste,  key=lambda c: CAIXA_ALVO[c][2] - CAIXA_ALVO[c][0])
CORES = {"alvo": "red", "outros": "deepskyblue"}

def painel(ax, cam, modelo, classes_aceitas, titulo):
    ax.imshow(np.array(Image.open(cam).convert("RGB")))
    for caixa, nome, conf in detectar(modelo, cam, classes_aceitas):
        x1, y1, x2, y2 = caixa; cor = CORES.get(nome, "yellow")
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=cor, linewidth=2))
        ax.text(x1, max(12, y1-5), f"{nome} {conf:.2f}", color=cor, fontsize=9, weight="bold")
    gt = CAIXA_ALVO[cam]
    ax.add_patch(patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                                   fill=False, edgecolor="lime", linewidth=1.5, linestyle="--"))
    ax.set_title(titulo, fontsize=11); ax.axis("off")

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
painel(axes[0,0], ref_longe, modelo_coco, CLASSES_MARITIMAS, "ANTES — alvo distante (COCO)")
painel(axes[0,1], ref_perto, modelo_coco, CLASSES_MARITIMAS, "ANTES — alvo proximo (COCO)")
painel(axes[1,0], ref_longe, modelo_alvo, None, "DEPOIS — alvo distante (especializado)")
painel(axes[1,1], ref_perto, modelo_alvo, None, "DEPOIS — alvo proximo (especializado)")
plt.suptitle("Verde tracejado = anotacao do MODD2 | vermelho = alvo | azul = outros", fontsize=13)
plt.tight_layout(); plt.savefig(SAIDAS / "antes_depois.png", dpi=120, bbox_inches="tight"); plt.show()

# Parte 7 — Da imagem para o mundo: calibracao estereo

Ate aqui tudo foi medido em pixels. O sistema de controle de um \ac{VSNT} precisa de outra coisa:
em que direcao o alvo esta e a que distancia. A ponte entre as duas e a geometria da camera.

Ha uma armadilha aqui, e ela vale a secao inteira. O \ac{MODD2} distribui as imagens **ja
retificadas**, mas os parametros de calibracao publicados sao os **brutos**. Usar a matriz
intrinseca original introduziria vies, porque a retificacao altera a distancia focal e o centro
optico. A celula abaixo refaz a retificacao sobre os parametros publicados, uma vez por sequencia,
e toma a matriz de projecao resultante — que e a que vale para as imagens efetivamente usadas.

A diferenca entre o valor bruto e o retificado aparece impressa, para que se veja o tamanho do erro
que se evitou.

In [ ]:
# ------------------------------------------------------------------
# Calibracao: parametros da imagem retificada
# ------------------------------------------------------------------
import cv2, numpy as np

def ler_calibracao(seq):
    """fx, cx, cy e base (m) validos para as imagens JA retificadas da sequencia."""
    caminho = CAMINHO_SEQ[seq].parent / "calibration.yaml"
    assert caminho.exists(), f"calibration.yaml nao encontrado em {caminho.parent}"
    fs = cv2.FileStorage(str(caminho), cv2.FILE_STORAGE_READ)
    M1, D1 = fs.getNode("M1").mat(), fs.getNode("D1").mat()
    M2, D2 = fs.getNode("M2").mat(), fs.getNode("D2").mat()
    R, T   = fs.getNode("R").mat(),  fs.getNode("T").mat()
    no = fs.getNode("imageSize")
    tam = ((int(no.at(0).real()), int(no.at(1).real())) if no.isSeq()
           else tuple(int(v) for v in no.mat().ravel()[:2]))
    fs.release()
    # as imagens ja estao retificadas; stereoRectify e chamado so para obter P1 e P2,
    # que trazem a distancia focal e o centro optico corretos depois da retificacao
    _, _, P1, P2, _, _, _ = cv2.stereoRectify(M1, D1, M2, D2, tam, R, T,
                                              flags=cv2.CALIB_ZERO_DISPARITY, alpha=0)
    fx, cx, cy = P1[0, 0], P1[0, 2], P1[1, 2]
    base_m = abs(P2[0, 3] / fx) / 1000.0        # T do MODD2 vem em milimetros
    return {"fx": float(fx), "cx": float(cx), "cy": float(cy), "base": float(base_m),
            "tam": tam, "fx_bruto": float(M1[0, 0]), "cx_bruto": float(M1[0, 2])}

CAL = ler_calibracao(SEQUENCIA_ALVO)

print("=" * 70)
print(f"CALIBRACAO — {SEQUENCIA_ALVO}")
print("=" * 70)
print(f"Imagem                 : {CAL['tam'][0]} x {CAL['tam'][1]}")
print(f"Retificada  fx / cx    : {CAL['fx']:.1f} px / {CAL['cx']:.1f} px")
print(f"Bruta (nao usar) fx/cx : {CAL['fx_bruto']:.1f} px / {CAL['cx_bruto']:.1f} px")
print(f"Base entre as cameras  : {CAL['base']:.4f} m")
print(f"Campo de visao horizontal: {2 * np.degrees(np.arctan(CAL['tam'][0] / 2 / CAL['fx'])):.1f} graus")
print(f"Resolucao angular no centro: {np.degrees(1 / CAL['fx']):.3f} graus por pixel")
print("-" * 70)
print("Erro de distancia para meio pixel de disparidade:")
for z in (25, 50, 75, 100):
    print(f"  a {z:3d} m  ->  +/- {z**2 / (CAL['fx'] * CAL['base']) * 0.5:5.1f} m")

## 7.1 — Como a distancia e obtida

A **marcacao** sai direto da coluna do centro da caixa: theta = arctan((u - cx) / fx). E geometria
simples, sem nenhuma suposicao sobre a embarcacao.

A **distancia** exige o par estereo. Como o conjunto nao anota a camera direita, a correspondencia
entre as duas vistas e estabelecida por **casamento de gabarito**: o recorte definido pela caixa
detectada na imagem esquerda e procurado na direita ao longo da mesma linha, com pequena tolerancia
vertical para absorver residuo de retificacao. O pico da correlacao normalizada fornece a
disparidade, refinada para fracao de pixel por ajuste de parabola nos tres pontos vizinhos ao pico.

Dai a distancia decorre de Z = fx * B / d, e a incerteza de dZ = Z^2 / (fx * B) * dd — que cresce
com o **quadrado** da distancia. E caracteristica da triangulacao, nao da implementacao, e e por
isso que a distancia e sempre reportada com a faixa de incerteza junto.

In [ ]:
# ------------------------------------------------------------------
# Marcacao e distancia
# ------------------------------------------------------------------
def marcacao_graus(u, cal=None):
    """Direcao de chegada em graus. Positiva para boreste, negativa para bombordo."""
    cal = cal or CAL
    return float(np.degrees(np.arctan((u - cal["cx"]) / cal["fx"])))

def caminho_par_direito(cam_esquerdo):
    """Caminho da imagem da camera direita correspondente."""
    p = Path(cam_esquerdo)
    return p.with_name(p.stem[:-1] + "R" + p.suffix)

def disparidade(cam_esquerdo, caixa, busca=BUSCA_DISPARIDADE, margem=TOLERANCIA_VERTICAL, pico_minimo=0.60):
    """Disparidade por casamento de gabarito ao longo da linha epipolar.

    Devolve (disparidade_px, desvio_vertical_px, pico_da_correlacao) ou None se
    o par direito nao existir, a caixa for degenerada ou a correlacao for fraca.
    """
    par = caminho_par_direito(cam_esquerdo)
    if not par.exists():
        return None
    L = cv2.imread(str(cam_esquerdo), cv2.IMREAD_GRAYSCALE)
    R = cv2.imread(str(par), cv2.IMREAD_GRAYSCALE)
    if L is None or R is None:
        return None

    x1, y1, x2, y2 = [int(round(v)) for v in caixa]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(L.shape[1], x2), min(L.shape[0], y2)
    if x2 - x1 < 8 or y2 - y1 < 6:
        return None
    gabarito = L[y1:y2, x1:x2]

    # a busca so precisa cobrir a faixa de disparidade positiva, a esquerda da caixa
    ya, yb = max(0, y1 - margem), min(L.shape[0], y2 + margem)
    xa, xb = max(0, x1 - busca), min(L.shape[1], x2 + margem)
    faixa = R[ya:yb, xa:xb]
    if faixa.shape[0] < gabarito.shape[0] or faixa.shape[1] < gabarito.shape[1]:
        return None

    res = cv2.matchTemplate(faixa, gabarito, cv2.TM_CCOEFF_NORMED)
    _, pico, _, loc = cv2.minMaxLoc(res)
    if pico < pico_minimo:
        return None

    # refino subpixel: parabola nos tres pontos ao redor do pico
    linha, i, sub = res[loc[1]], loc[0], 0.0
    if 0 < i < len(linha) - 1:
        den = linha[i - 1] - 2 * linha[i] + linha[i + 1]
        if abs(den) > 1e-6:
            sub = float(np.clip(0.5 * (linha[i - 1] - linha[i + 1]) / den, -1, 1))
    return x1 - (xa + i + sub), int(loc[1] - (y1 - ya)), float(pico)

def distancia_m(disp, cal=None):
    cal = cal or CAL
    return cal["fx"] * cal["base"] / disp if disp and disp > 0.2 else float("nan")

def incerteza_m(z, erro_disp=0.5, cal=None):
    cal = cal or CAL
    return (z ** 2) / (cal["fx"] * cal["base"]) * erro_disp

print("[ok] Funcoes de marcacao e distancia prontas.")

# Parte 8 — Marcacao e distancia ao longo da aproximacao

A partir daqui **nenhuma anotacao do conjunto e usada para estimar**: a caixa empregada e a que o
detector produziu, como ocorreria a bordo. As anotacoes voltam apenas como referencia, para medir o
erro angular.

Em cada quadro do percurso toma-se a deteccao da classe `alvo` de maior confianca. Nao ha rastreador:
a propria classificacao resolve a identidade, e a matriz de confusao da Parte 6 e o que autoriza essa
escolha. Quadro em que o alvo nao e detectado simplesmente nao produz estimativa — a serie fica com
um buraco, que e mais honesto do que interpolar.

In [ ]:
# ------------------------------------------------------------------
# Serie de parametros do alvo ao longo do percurso de teste
# ------------------------------------------------------------------
cams_percurso = sorted(cam_teste, key=lambda c: int(Path(c).stem[:-1]))
print(f"Percorrendo {len(cams_percurso)} quadros do teste ...\n")

linhas, sem_alvo = [], 0
for cam in cams_percurso:
    idx = int(Path(cam).stem[:-1])
    alvos = [d for d in detectar(modelo_alvo, cam) if d[1] == "alvo"]
    if not alvos:
        sem_alvo += 1
        continue
    caixa, _, conf = max(alvos, key=lambda d: d[2])
    u = (caixa[0] + caixa[2]) / 2
    reg = {"quadro": idx, "conf": round(conf, 3), "u": round(u, 1),
           "largura_px": round(caixa[2] - caixa[0], 1),
           "marcacao_deg": round(marcacao_graus(u), 3)}
    d = disparidade(cam, caixa)
    if d:
        disp, dv, pico = d
        z = distancia_m(disp)
        reg.update({"disparidade_px": round(disp, 2), "pico_correlacao": round(pico, 2),
                    "desvio_vertical_px": dv, "distancia_m": round(z, 1),
                    "incerteza_m": round(incerteza_m(z), 1),
                    # comprimento implicito: so para checar coerencia, nunca para estimar
                    "comprimento_implicito_m": round(z * (caixa[2]-caixa[0]) / CAL["fx"], 2)})
    gt = CAIXA_ALVO.get(cam)
    if gt is not None:
        reg["marcacao_gt_deg"] = round(marcacao_graus((gt[0] + gt[2]) / 2), 3)
    linhas.append(reg)

df_alvo = pd.DataFrame(linhas)
df_alvo.to_csv(SAIDAS / "parametros_alvo.csv", index=False)
validas = df_alvo.dropna(subset=["distancia_m"]) if "distancia_m" in df_alvo else df_alvo.iloc[0:0]

print("=" * 74); print("PARAMETROS DO ALVO"); print("=" * 74)
print(f"Quadros do percurso        : {len(cams_percurso)}")
print(f"Com deteccao da classe alvo: {len(df_alvo)}  ({len(df_alvo)/len(cams_percurso):.0%})")
print(f"Sem deteccao               : {sem_alvo}")
print(f"Com distancia estimada     : {len(validas)}")
if len(validas):
    print(f"\nDistancia  : {validas['distancia_m'].max():.0f} m -> {validas['distancia_m'].min():.0f} m")
    print(f"Incerteza  : +/- {validas['incerteza_m'].max():.0f} m a +/- {validas['incerteza_m'].min():.0f} m")
    c = validas["comprimento_implicito_m"]
    print(f"Comprimento implicito: mediana {c.median():.2f} m | desvio {c.std():.2f} m")
    print("  (a lancha nao muda de tamanho; a estabilidade desse numero e o que sustenta a")
    print("   distancia na ausencia de referencia no conjunto)")
if "marcacao_gt_deg" in df_alvo:
    err = (df_alvo["marcacao_deg"] - df_alvo["marcacao_gt_deg"]).abs().dropna()
    if len(err):
        print(f"\nErro de marcacao contra a anotacao: media {err.mean():.2f} | maximo {err.max():.2f} graus")
print(f"\nCSV: {SAIDAS / 'parametros_alvo.csv'}")

## 8.1 — As tres series

O primeiro grafico e a **marcacao**, comparada a que sairia da caixa anotada. O segundo e a
**distancia** com a faixa de incerteza sombreada — repare como a faixa se estreita conforme o alvo
se aproxima, que e o comportamento previsto pela formula. O terceiro e o **comprimento implicito**:
como a lancha nao muda de tamanho, essa curva deveria ser plana, e o quanto ela oscila e a medida
pratica da qualidade da estimativa.

In [ ]:
# ------------------------------------------------------------------
# Tres graficos: marcacao, distancia com incerteza e coerencia
# ------------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

ax = axes[0]
ax.plot(df_alvo["quadro"], df_alvo["marcacao_deg"], "-o", ms=3, color=AZUL, label="estimada")
if "marcacao_gt_deg" in df_alvo:
    ax.plot(df_alvo["quadro"], df_alvo["marcacao_gt_deg"], "--", color="#52514e", lw=1.2,
            label="da anotacao")
ax.axhline(0, color="#999", lw=0.8)
ax.set_ylabel("marcacao (graus)"); ax.legend(frameon=False)
ax.set_title("Marcacao do alvo  (positiva para boreste)", fontsize=11)

ax = axes[1]
if len(validas):
    z, e = validas["distancia_m"], validas["incerteza_m"]
    ax.fill_between(validas["quadro"], z - e, z + e, color=LARANJA, alpha=0.22, label="incerteza")
    ax.plot(validas["quadro"], z, "-o", ms=3, color=LARANJA, label="distancia")
    ax.legend(frameon=False)
ax.set_ylabel("distancia (m)")
ax.set_title("Distancia por triangulacao estereo, com a incerteza associada", fontsize=11)

ax = axes[2]
if len(validas):
    c = validas["comprimento_implicito_m"]
    ax.plot(validas["quadro"], c, "-o", ms=3, color=AGUA)
    ax.axhline(c.median(), color="#52514e", ls="--", lw=1.2,
               label=f"mediana {c.median():.2f} m")
    ax.legend(frameon=False)
ax.set_ylabel("comprimento implicito (m)"); ax.set_xlabel("quadro")
ax.set_title("Checagem de coerencia: a lancha nao muda de tamanho", fontsize=11)

for a in axes:
    a.grid(alpha=0.18)
    for lado in ("top", "right"):
        a.spines[lado].set_visible(False)
plt.tight_layout(); plt.savefig(SAIDAS / "parametros_alvo.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ------------------------------------------------------------------
# Um quadro com os parametros estimados sobrepostos
# ------------------------------------------------------------------
if len(validas):
    alvo_q = int(validas.iloc[len(validas) // 2]["quadro"])
    cam = next(c for c in cams_percurso if int(Path(c).stem[:-1]) == alvo_q)
    linha = validas[validas["quadro"] == alvo_q].iloc[0]
    det = [d for d in detectar(modelo_alvo, cam) if d[1] == "alvo"]
    caixa = max(det, key=lambda d: d[2])[0]

    fig, ax = plt.subplots(figsize=(13, 9))
    ax.imshow(np.array(Image.open(cam).convert("RGB")))
    x1, y1, x2, y2 = caixa
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=2.5))
    ax.axvline(CAL["cx"], color="yellow", ls=":", lw=1.2)
    ax.text(CAL["cx"] + 6, 30, "eixo optico", color="yellow", fontsize=9)
    txt = (f"marcacao {linha['marcacao_deg']:+.2f} graus\n"
           f"distancia {linha['distancia_m']:.0f} +/- {linha['incerteza_m']:.0f} m\n"
           f"largura {linha['largura_px']:.0f} px | confianca {linha['conf']:.2f}")
    ax.text(x1, max(30, y1 - 14), txt, color="white", fontsize=11, va="bottom",
            bbox=dict(boxstyle="round,pad=0.4", fc="black", alpha=0.65))
    ax.set_title(f"Quadro {alvo_q} — o que o sistema embarcado entregaria", fontsize=12)
    ax.axis("off")
    plt.tight_layout(); plt.savefig(SAIDAS / "quadro_anotado.png", dpi=130, bbox_inches="tight"); plt.show()

# Parte 9 — Resumo, zip e download

Esta celula reune tudo o que foi produzido, imprime os numeros principais e dispara o download.
As pastas do Colab somem quando a sessao encerra, entao ela e a ultima a rodar — e a unica que
precisa ser rodada de novo se voce reabrir o notebook depois.

In [ ]:
# ------------------------------------------------------------------
# Resumo + zip + download
# ------------------------------------------------------------------
import datetime

for nome in ("results.png", "results.csv", "confusion_matrix.png", "labels.jpg", "args.yaml"):
    achados = sorted(TREINOS.rglob(nome), key=lambda p: p.stat().st_mtime)
    if achados:
        shutil.copy2(achados[-1], SAIDAS / f"treino_{nome}")

print("=" * 78); print("RESUMO DA EXECUCAO"); print("=" * 78)
def linha(rotulo, valor):
    print(f"{rotulo:24s}: {valor}")
try:
    linha("Configuracao", f"{Path(PESOS).stem} | {IMGSZ} px | {EPOCAS} epocas | escala {ESCALA_AUG}")
    linha("Alvo", f"{SEQUENCIA_ALVO} | {len(quadros_alvo)} quadros | "
                  f"{min(larguras):.0f}-{max(larguras):.0f} px")
    linha("Dataset", f"{contagem['train']} treino / {contagem['valid']} valid / {contagem['test']} teste")
    linha("Augmentation", f"{len(VARIANTES)} tecnicas ({', '.join(rotulo_var(t, v) for t, v in VARIANTES)})"
                          f" | {total_aug} imagens, so no treino")
except Exception:
    pass
try:
    linha("mAP@50 / mAP@50-95", f"{metricas.box.map50:.3f} / {metricas.box.map:.3f}")
    linha("Antes  (COCO)", f"P {antes_full['P']:.3f} | R {antes_full['R']:.3f} | F1 {antes_full['F1']:.3f}")
    linha("Depois (especialista)", f"P {depois_full['P']:.3f} | R {depois_full['R']:.3f} | F1 {depois_full['F1']:.3f}")
    linha("Ganho de F1", f"{depois_full['F1'] - antes_full['F1']:+.3f}")
except Exception:
    pass
try:
    linha("Matriz de confusao", f"alvo->alvo {M[0,0]} | alvo->outros {M[0,1]} | nao detectado {M[0,2]}")
    linha("Falso positivo em fundo", f"{M[2,0]} como alvo, {M[2,1]} como outros")
except Exception:
    pass
try:
    if len(validas):
        linha("Distancia estimada", f"{validas['distancia_m'].max():.0f} m -> {validas['distancia_m'].min():.0f} m")
        linha("Comprimento implicito", f"mediana {validas['comprimento_implicito_m'].median():.2f} m | "
                                       f"desvio {validas['comprimento_implicito_m'].std():.2f} m")
    err = (df_alvo["marcacao_deg"] - df_alvo["marcacao_gt_deg"]).abs().dropna()
    if len(err):
        linha("Erro de marcacao", f"media {err.mean():.2f} | maximo {err.max():.2f} graus")
except Exception:
    pass
print("=" * 78)

carimbo = datetime.datetime.now().strftime("%d%b_%H%M").lower()
nome_zip = f"saidas_tcc_{TAG}_{carimbo}" if "TAG" in dir() else f"saidas_tcc_{carimbo}"
caminho_zip = shutil.make_archive(f"/content/{nome_zip}", "zip", SAIDAS)
print(f"\nArquivos em {SAIDAS}: {len(list(SAIDAS.iterdir()))}")
print(f"Zip: {caminho_zip}  ({Path(caminho_zip).stat().st_size / 1e6:.1f} MB)")

if USAR_DRIVE and PASTA_DRIVE is not None:
    shutil.copy2(caminho_zip, PASTA_DRIVE / Path(caminho_zip).name)
    print(f"[drive] copia em {PASTA_DRIVE / Path(caminho_zip).name}")

try:
    from google.colab import files
    files.download(caminho_zip)
    print("\nDownload disparado. Se o navegador bloquear, o zip esta no Drive.")
except Exception as e:
    print(f"\n[info] download automatico indisponivel ({e}). Baixe pelo painel de arquivos.")